# Paper 25: R3M: A Universal Visual Representation for Robot Manipulation
**Authors**: Suraj Nair, Aravind Rajeswaran, Vikash Kumar, Chelsea Finn, Abhinav Gupta (Meta AI, Stanford, NYU - NeurIPS 2022)

---

## 1. The Historical Breaking Point: The Robot Visual Generalization Gap
In robot manipulation, training policies from raw pixels typically fails when camera angles, background distractors, or lighting vary. Standard ImageNet representations fail because they prioritize static category labels while discarding **spatial positions, physical hand-object interactions, and temporal progress**.

## 2. The Thought Experiment: Learning from Egocentric Human Videos
R3M pre-trains visual encoders on egocentric human activity videos (Ego4D) using three unified self-supervised objectives:
1. **Time-Contrastive Learning (TCN)**: Adjacent video frames ($|t - t'| \le \delta$) are pulled together, while temporally distant frames and frames from other videos act as negative distractors.
2. **Language-Image Alignment (CLIP)**: Video frames are contrastively aligned with natural language task narrations (e.g. *"picking up the yellow block"*).
3. **$L_1$ Sparsity Regularization**: Penalizes representation coordinates to prevent unbounded feature drift and ensure compact manifolds for downstream behavior cloning.

$$\mathcal{L}_{\text{R3M}} = \lambda_{\text{tcn}} \mathcal{L}_{\text{TCN}} + \lambda_{\text{lang}} \mathcal{L}_{\text{lang}} + \lambda_{\text{reg}} \|z\|_1$$

In this notebook, you will explore:
1. Temporal representation manifolds and video frame alignment
2. Multi-task training of R3M
3. Downstream robot policy learning (Behavior Cloning) using frozen R3M features

---
## Setup & Autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from model import (
    R3MTextEncoder,
    R3M,
)

print(f"PyTorch version: {torch.__version__}")

---
## 1. Time-Contrastive Learning (TCN) Dynamics
Let's simulate a temporal manipulation trajectory (e.g. a robot arm reaching toward an object). Notice how adjacent frames are mapped to nearby positions in embedding space.

In [ ]:
torch.manual_seed(42)
T_steps = 10
# Simulate a trajectory of 10 sequential video frames
trajectory = torch.randn(T_steps, 3, 16, 16)
# Add smooth temporal drift
for t in range(1, T_steps):
    trajectory[t] = 0.8 * trajectory[t - 1] + 0.2 * trajectory[t]

model = R3M(embed_dim=32, feat_dim=64, vocab_size=100)
with torch.no_grad():
    embeds = model.encode_image(trajectory)
    sim_temporal = (embeds @ embeds.T).cpu().numpy()

plt.figure(figsize=(5, 4))
plt.imshow(sim_temporal, cmap="viridis")
plt.colorbar(label="Cosine Similarity")
plt.title("Temporal Video Frame Similarity Matrix")
plt.xlabel("Timestep t'")
plt.ylabel("Timestep t")
plt.show()

---
## 2. Multi-Task Training of R3M
We train R3M simultaneously with Time-Contrastive Learning, Language Alignment, and L1 Regularization.

In [ ]:
torch.manual_seed(42)
B = 4
model = R3M(embed_dim=32, feat_dim=64, vocab_size=100, temperature=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)

base_frames = torch.randn(B, 3, 16, 16)
im_anchor = base_frames + 0.03 * torch.randn_like(base_frames)
im_pos = base_frames + 0.03 * torch.randn_like(base_frames)

narrations = torch.tensor([
    [1, 2, 3, 4],     # "reach for red block"
    [10, 20, 30, 40], # "pick up blue block"
    [50, 60, 70, 80], # "open the drawer"
    [85, 90, 95, 99], # "push the green cup"
], dtype=torch.long)

tcn_losses, lang_losses, total_losses = [], [], []

for step in range(60):
    optimizer.zero_grad()
    total_loss, l_tcn, l_lang, l_reg = model(im_anchor, im_pos, narrations)
    total_loss.backward()
    optimizer.step()

    tcn_losses.append(l_tcn.item())
    lang_losses.append(l_lang.item())
    total_losses.append(total_loss.item())

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(total_losses, color="crimson")
axes[0].set_title("R3M Total Loss")
axes[0].set_xlabel("Step")
axes[0].grid(True, alpha=0.3)

axes[1].plot(tcn_losses, color="teal")
axes[1].set_title("Time-Contrastive Loss (TCN)")
axes[1].set_xlabel("Step")
axes[1].grid(True, alpha=0.3)

axes[2].plot(lang_losses, color="purple")
axes[2].set_title("Language Grounding Loss")
axes[2].set_xlabel("Step")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Robot Policy Learning: Behavior Cloning on Frozen R3M Features
The primary benchmark for robotic representations: freeze R3M as a visual backbone and train a lightweight 2-layer MLP policy to predict robot end-effector velocities $(\Delta x, \Delta y, \Delta z, \text{gripper})$.

In [ ]:
# Freeze R3M visual encoder
model.eval()
with torch.no_grad():
    frozen_features = model.encode_image(base_frames)  # (B, embed_dim)

# Target robot action: (dx, dy, dz, gripper_state)
target_actions = torch.tensor([
    [ 0.1,  0.0, -0.05, 1.0],
    [-0.05, 0.2,  0.0,  0.0],
    [ 0.0, -0.1,  0.1,  1.0],
    [ 0.2,  0.0,  0.0,  0.0],
])

# Lightweight MLP Robot Policy
policy = nn.Sequential(
    nn.Linear(model.embed_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 4),
)
policy_opt = torch.optim.AdamW(policy.parameters(), lr=1e-2)

policy_losses = []
for _ in range(50):
    policy_opt.zero_grad()
    pred_actions = policy(frozen_features)
    loss_action = F.mse_loss(pred_actions, target_actions)
    loss_action.backward()
    policy_opt.step()
    policy_losses.append(loss_action.item())

print(f"Initial Action MSE: {policy_losses[0]:.4f} -> Final Action MSE: {policy_losses[-1]:.6f}")
plt.figure(figsize=(6, 3))
plt.plot(policy_losses, color="darkgreen")
plt.title("Behavior Cloning Policy Convergence on Frozen R3M Features")
plt.xlabel("Policy Training Step")
plt.ylabel("Action MSE Loss")
plt.grid(True, alpha=0.3)
plt.show()